# Baseline Training & Fairness

1. **Data Preparation**
   - Load and preprocess combined data
   - Train-test-split
   - Feature engineering

2. **Model Training**
   - Train Random Forest
   - Hyperparameter tuning (optional)
   - Cross-validation

3. **Performance Evaluation**
   - Accuracy, Precision, Recall, F1, AUC
   - Confusion matrix
   - Per-group performance

4. **Fairness Analysis**
   - Evaluate fairness metrics (AIF360)
   - Demographic parity
   - Equalized odds
   - Disparate impact

5. **Bias Mitigation**
   - Apply reweighing (preprocessing)
   - Compare fair vs unfair model
   - Trade-off analysis

6. **Explainability**
   - SHAP feature importance
   - Summary plots
   - Identify bias sources

# Import

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import load_all_banks, preprocess_data
from src.models import train_random_forest, tune_hyperparameters
from src.evaluation import evaluate_model, evaluate_fairness, explain_model_shap
from sklearn.model_selection import train_test_split

import numpy as np

In [3]:
# Load data
combined_df, _ = load_all_banks(data_path='../data/raw')

Loading bank data...
✓ Loaded BankA: 226456 samples
✓ Loaded BankB: 402923 samples
✓ Loaded BankC: 170620 samples

✓ Total combined: 799999 samples
  Distribution: {'Bank B': 402923, 'Bank A': 226456, 'Bank C': 170620}


In [4]:
# Split
train_df, test_df = train_test_split(combined_df, test_size=0.2, random_state=42)

In [5]:
# Preprocess
X_train, y_train, artifacts = preprocess_data(train_df, missing_strategy='combined', fit_transform=True)
X_test, y_test, _ = preprocess_data(test_df, missing_strategy='combined', scaler=artifacts['scaler'], 
                                     encoder=artifacts['encoder'], fit_transform=False)


Preprocessing data...
  Filled workclass with mode: Private
  Filled occupation with mode: Prof-specialty
  Filled native-country with mode: United-States
  ✓ Fitted StandardScaler on 5 numerical features
  ✓ Fitted OneHotEncoder: 8 → 99 features
  ✓ Final shape: (639999, 104)
  ✓ Class distribution: [481276 158723]

Preprocessing data...
  Filled workclass with mode: Private
  Filled occupation with mode: Prof-specialty
  Filled native-country with mode: United-States
  ✓ Final shape: (160000, 104)
  ✓ Class distribution: [120255  39745]


# Training Baseline Model

In [ ]:
# Hyperparameter Tuning

param_grid = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
} # from this grid random values are taken

model, best_params = tune_hyperparameters(
    X_train, y_train,
    param_grid=param_grid,
    method='random', # instead of 'grid' because of the amount of time it takes
    cv=3,  # Number of cross-validation folds
    verbose=2
)

# best_params = {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 30}



Hyperparameter Tuning (RANDOM search)...
  Parameter grid: {'n_estimators': [50, 100, 150, 200, 300], 'max_depth': [10, 15, 20, 25, 30], 'min_samples_split': [2, 5, 10, 15], 'min_samples_leaf': [1, 2, 4, 6]}
  Cross-validation folds: 3
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=25, min_samples_leaf=4, min_samples_split=2, n_estimators=50; total time= 4.9min
[CV] END max_depth=25, min_samples_leaf=4, min_samples_split=2, n_estimators=50; total time= 5.0min
[CV] END max_depth=25, min_samples_leaf=4, min_samples_split=2, n_estimators=50; total time= 5.1min
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=24.8min
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=24.9min
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time= 9.8min
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=24.9m

In [9]:
model = train_random_forest(
    X_train,
    y_train,
    best_params,
    random_state = 42
)


Training Random Forest...
  Parameters: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 30}
✓ Training complete!


# Evaluation

### Performance

In [10]:
results = evaluate_model(model, X_test, y_test, detailed = True)


Evaluating model...

Performance Metrics:
  Accuracy:  0.8604
  Precision: 0.7859
  Recall:    0.6018
  F1-Score:  0.6816
  ROC-AUC:   0.9141

Confusion Matrix:
  TN: 113741  FP:  6514
  FN: 15828  TP: 23917

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91    120255
           1       0.79      0.60      0.68     39745

    accuracy                           0.86    160000
   macro avg       0.83      0.77      0.80    160000
weighted avg       0.86      0.86      0.85    160000



### Fariness-Metrics

In [13]:
test_df_original = test_df.copy()

gender_map = {'Male': 1, 'Female': 0}
race_map = {'White': 1}  # Others will be 0

gender_test = test_df_original['gender'].map(gender_map).fillna(0).values
race_test = np.where(test_df_original['race'] == 'White', 1, 0)

fairness_results_gender = evaluate_fairness(
    y_test,
    results['predictions'],
    sensitive_features={'gender': gender_test},
    privileged_groups=[{'gender': 1}],
    unprivileged_groups=[{'gender': 0}]
)

fairness_results_race = evaluate_fairness(
    y_test,
    results['predictions'],
    sensitive_features={'race': race_test},
    privileged_groups=[{'race': 1}],
    unprivileged_groups=[{'race': 0}]
)

fairness_results = {
    'gender': fairness_results_gender,
    'race': fairness_results_race
}


Evaluating Fairness Metrics...

Fairness Metrics:
  Disparate Impact:         0.3051 (>= 0.8 is fair)
  Statistical Parity Diff:  -0.1740 (0 is fair)
  Equal Opportunity Diff:   -0.1033 (0 is fair)
  Average Odds Diff:        -0.0811 (0 is fair)
  Theil Index:              0.1199 (lower is fairer)

⚠ Model does NOT meet fairness criteria

Evaluating Fairness Metrics...

Fairness Metrics:
  Disparate Impact:         0.5574 (>= 0.8 is fair)
  Statistical Parity Diff:  -0.0898 (0 is fair)
  Equal Opportunity Diff:   -0.0877 (0 is fair)
  Average Odds Diff:        -0.0572 (0 is fair)
  Theil Index:              0.1199 (lower is fairer)

⚠ Model does NOT meet fairness criteria


# New Model with Bias-Mitigation

### Bias Mitigation

In [ ]:
# 1. extract sensitive features for train data
train_df_original = train_df.copy()

gender_train = train_df_original['gender'].map(gender_map).fillna(0).values
race_train = np.where(train_df_original['race'] == 'White', 1, 0)

# 2. Bias Mitigation Reweighing (only on training data)
from src.evaluation.fairness import mitigate_bias

X_train_fair, y_train_fair, sample_weights = mitigate_bias(
    X_train, 
    y_train,
    sensitive_features_train={
        'gender': gender_train,
        'race': race_train  # both sensitive features
    },
    method='reweighing',
    privileged_groups=[{'gender': 1, 'race': 1}],     # White Male
    unprivileged_groups=[{'gender': 0, 'race': 0}]    # Non-White Female
)


Applying bias mitigation: reweighing...
✓ Reweighing applied
  Weight range: [0.756, 2.958]


In [ ]:
# remove the last 2 columns (gender, race) again
X_train_fair = X_train_fair[:, :-2]  # 106 → 104 Features

print(f"✓ X_train_fair shape: {X_train_fair.shape}")  # should be (n, 104)
print(f"✓ X_test shape: {X_test.shape}")              # (n, 104)

✓ X_train_fair shape: (639999, 104)
✓ X_test shape: (160000, 104)


### Training

In [26]:
# Train a fair model


model_fair = train_random_forest(
    X_train_fair,
    y_train_fair,
    best_params,
    random_state = 42
)


Training Random Forest...
  Parameters: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 30}
✓ Training complete!


### Evaluation: Performance

In [27]:
results_fair = evaluate_model(model_fair, X_test, y_test, detailed = True)


Evaluating model...

Performance Metrics:
  Accuracy:  0.8604
  Precision: 0.7861
  Recall:    0.6017
  F1-Score:  0.6816
  ROC-AUC:   0.9140

Confusion Matrix:
  TN: 113750  FP:  6505
  FN: 15832  TP: 23913

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91    120255
           1       0.79      0.60      0.68     39745

    accuracy                           0.86    160000
   macro avg       0.83      0.77      0.80    160000
weighted avg       0.86      0.86      0.85    160000



### Evaluation: Fairness

In [28]:
fairness_results_gender = evaluate_fairness(
    y_test,
    results_fair['predictions'],
    sensitive_features={'gender': gender_test},
    privileged_groups=[{'gender': 1}],
    unprivileged_groups=[{'gender': 0}]
)

fairness_results_race = evaluate_fairness(
    y_test,
    results_fair['predictions'],
    sensitive_features={'race': race_test},
    privileged_groups=[{'race': 1}],
    unprivileged_groups=[{'race': 0}]
)


Evaluating Fairness Metrics...

Fairness Metrics:
  Disparate Impact:         0.3078 (>= 0.8 is fair)
  Statistical Parity Diff:  -0.1730 (0 is fair)
  Equal Opportunity Diff:   -0.1013 (0 is fair)
  Average Odds Diff:        -0.0797 (0 is fair)
  Theil Index:              0.1199 (lower is fairer)

⚠ Model does NOT meet fairness criteria

Evaluating Fairness Metrics...

Fairness Metrics:
  Disparate Impact:         0.5603 (>= 0.8 is fair)
  Statistical Parity Diff:  -0.0892 (0 is fair)
  Equal Opportunity Diff:   -0.0864 (0 is fair)
  Average Odds Diff:        -0.0563 (0 is fair)
  Theil Index:              0.1199 (lower is fairer)

⚠ Model does NOT meet fairness criteria


In [ ]:
# Evaluate fairness metrics
fairness_fair_gender = evaluate_fairness(
    y_test, results_fair['predictions'],
    sensitive_features={'gender': gender_test},
    privileged_groups=[{'gender': 1}],
    unprivileged_groups=[{'gender': 0}]
)

fairness_fair_race = evaluate_fairness(
    y_test, results_fair['predictions'],
    sensitive_features={'race': race_test},
    privileged_groups=[{'race': 1}],
    unprivileged_groups=[{'race': 0}]
)